# 04 - Create Reward Preference Pair Dataset

This notebook creates matched plain/OCN response variants, verifies detector separation, saves to Drive, and publishes the pair dataset to Hugging Face.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, textwrap

DEFAULT_REPO_URL = "https://github.com/ritwikraha/AutoRegressive-Bhasha.git"

def find_repo_root():
    try:
        import google.colab  # type: ignore  # noqa: F401
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception:
        pass

    candidates = [
        Path.cwd(),
        Path("/content/empty-negations"),
        Path("/content/AutoRegressive-Bhasha/empty-negations"),
        Path("/content/drive/MyDrive/ocn_empty_negations"),
        Path("/content/drive/MyDrive/AutoRegressive-Bhasha/empty-negations"),
    ]
    for candidate in candidates:
        if (candidate / "src/ocn").exists():
            return candidate
    repo_url = os.environ.get("OCN_REPO_URL", DEFAULT_REPO_URL)
    target = Path("/content/AutoRegressive-Bhasha")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)

    cloned_candidates = [target / "empty-negations", target]
    for candidate in cloned_candidates:
        if (candidate / "src/ocn").exists():
            return candidate

    raise FileNotFoundError(
        f"Cloned {repo_url}, but could not find src/ocn. "
        "Set OCN_REPO_URL to a repository containing empty-negations/src/ocn."
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import wandb

from ocn.colab_utils import login_huggingface, login_wandb, make_colab_paths, publish_dataframe_to_hf, save_dataframe
from ocn.detectors import OCNDetector
from ocn.reward_pairs import PropositionSet, starter_proposition_sets, variants_to_frame

paths = make_colab_paths()
config = json.loads((paths.project_root / "ocn_colab_config.json").read_text())
login_huggingface("HF_WRITE_ACCESS")
run = login_wandb(project="ocn-empty-negations", name=f"reward-pairs-{config['run_id']}", config=config)

In [ ]:
extra_items = [
    PropositionSet("r004", "remote work policy", "The policy", "improves hiring flexibility", "changes coordination costs across teams"),
    PropositionSet("r005", "API design", "The design", "makes endpoints easier to use", "reduces integration errors for developers"),
    PropositionSet("r006", "data privacy", "The program", "protects customer information", "strengthens trust with users and regulators"),
    PropositionSet("r007", "process improvement", "The change", "reduces manual review time", "gives teams clearer visibility into bottlenecks"),
    PropositionSet("r008", "CRISPR", "The technique", "edits targeted genetic sequences", "creates new possibilities for biological research"),
    PropositionSet("r009", "budgeting app", "The app", "tracks spending", "helps users notice patterns before they become problems"),
    PropositionSet("r010", "printing press", "The invention", "increased copying speed", "changed how knowledge circulated across institutions"),
    PropositionSet("r011", "leadership", "Leadership", "coordinates group action", "helps people make decisions under uncertainty"),
    PropositionSet("r012", "sincere apology", "A sincere apology", "acknowledges harm", "creates conditions for repair"),
]

pairs = variants_to_frame(starter_proposition_sets() + extra_items, shuffle=True, seed=42)
scored_pairs = OCNDetector().annotate_rows(pairs, text_column="response")
scored_pairs.groupby("variant_type")[["has_ocn", "ocn_count"]].mean()

In [ ]:
pair_path = save_dataframe(scored_pairs, Path(config["drive_data_root"]) / "ocn_reward_pairs.csv")
repo_url = publish_dataframe_to_hf(
    scored_pairs,
    repo_id=config["hf_reward_pairs_repo"],
    split="train",
    private=config["hf_private"],
    card_path=REPO_ROOT / "dataset_cards/ocn_reward_pairs.md",
    commit_message=f"Publish OCN reward pairs {config['run_id']}",
)
wandb.log({
    "reward_pair_rows": len(scored_pairs),
    "reward_pairs": wandb.Table(dataframe=scored_pairs),
})
run.finish()
print("Saved:", pair_path)
print("Published:", repo_url)